# Notebook Metadata Bootstrap Example

This notebook demonstrates how to use the `dd_cleaner.notebook_utils` APIs to initialize a notebook session, discover available artifacts, and expose dataset bootstrap metadata through the metadata authority table.


In [ ]:
import sys
from pathlib import Path

config_file_name = 'itsm_config.yaml'

# Ensure we import from the local repository source tree, not an installed package.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = next((p for p in candidates if (p / config_file_name).exists()), None)
if repo_root is None:
    raise FileNotFoundError(f'Could not locate {config_file_name} in the expected repository paths.')

sys.path.insert(0, str(repo_root / 'src'))
from dd_cleaner.notebook_utils import init_notebook_session, get_metadata_table, get_dataset_metadata

working_dir = repo_root
config_path = repo_root / config_file_name

print('Using workspace:', working_dir)
print('Using config:', config_path)


In [ ]:
coord, artifacts = init_notebook_session(str(working_dir), config_path=str(config_path))

print('Available artifacts:')
display(artifacts)


In [ ]:
if coord.synchronized_dictionary_path.exists():
    df_metadata = get_metadata_table(coord)
    dataset_metadata = get_dataset_metadata(coord)

    print('Dataset-level bootstrap metadata (separate artifact):')
    display(dataset_metadata)

    print('\nPer-attribute metadata authority table (row-level):')
    print(list(df_metadata.columns))
    display(df_metadata.head())
else:
    print('The cleaner baseline has not been established yet.')
    print('Please run the cleaner pipeline before calling get_metadata_table().')
    print('Example command:')
    print(f'  uv run clean-dataset --config {config_path} --action full')
    print('Then rerun this cell.')


## Bootstrap metadata fields exposed by the notebook API

The `get_metadata_table()` function bootstraps the authoritative metadata table from the cleaner's synchronized dictionary and enriches it with dataset bootstrap metadata from `config.yaml`, including fields such as: `dataset_type`, `subject`, `wide_short_homogeneous`, `wide_short_representative_column`, and use-case answers.
